In [19]:
# STEP 1: Load the external knowledgebase and convert into pages.
# Example - Knowledgebase in pdf format
from langchain_community.document_loaders import PyPDFLoader
loader =PyPDFLoader('hr-leave-policy.pdf')
pdf_pages = loader.load()
print(f'No of pages: {len(pdf_pages)}')

No of pages: 5


In [21]:
# STEP 2: Split pages into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Spilit pages into small chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ",", " "]
)
split_docs = splitter.split_documents(pdf_pages)
print(f'Total no of chunks {len(split_docs)}')

Total no of chunks 36


In [34]:
# STEP 3: Create vector embeddings using FAISS
#py -m pip install faiss-cpu    
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
vector_store=FAISS.from_documents(split_docs, embedding=embedding_model)
vector_store.save_local('faiss_leave_policy')
print("Vector Store created ")

Vector Store created 


In [33]:
# STEP 4: Create vector embedding using Chroma
#py -m pip install chromadb langchain-chroma
from langchain_community.vectorstores import Chroma
persist_directory="chroma_leave_policy"
vector_store=Chroma.from_documents(documents=split_docs,embedding=embedding_model,persist_directory=persist_directory)
print('HR Leave vector store is created using Chroma...')

HR Leave vector store is created using Chroma...


In [36]:
# STEP 4: Build Retrieval Chain

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print("Model is ready")

USER_PROMPT_TEMPLATE = """Use the following pieces of the context to answer user's question.
If you don't know the answer, just say you don't know, don't try to make up the answer.
------------------------
{context}
Question: {question}
"""

USER_PROMPT = PromptTemplate(
    template=USER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
qa_chain = RetrievalQA.from_chain_type(
    model,
    chain_type="stuff",
    retriever=vector_store.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": USER_PROMPT}
)
print(qa_chain)
print('Retrieval chain is built successfully...')

Model is ready
verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of the context to answer user's question.\nIf you don't know the answer, just say you don't know, don't try to make up the answer.\n------------------------\n{context}\nQuestion: {question}\n"), llm=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='gpt-oss:120b-cloud'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context') return_source_documents=True retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020AFE519F90>, search_kwargs={})
Retrieval